# Section 2 — Cache Frame Features

> Design: `PIPELINE.md`. Previous: `1_choose_model` → `ResNet50_10C.pth`.

Run the **frozen** ResNet50 over every clip **once** and save the per-frame features.

```
Diving_01.pt   (16, 3, 224, 224)   9.63 MB
      │  frozen ResNet50 — one time
      ▼
Diving_01      (16, 2048)          65.5 KB      147x smaller
```

## Why

After Section 1 the ResNet is frozen forever — it returns the *same answer for the same
clip* every time. Recomputing it every epoch is pure waste.

| | per epoch |
|---|---|
| without cache | ~80 s (ResNet over 15,904 frames) |
| with cache | ~2 s (LSTM only) |

**40x faster** — which is what makes Section 5's 12 ablation runs a coffee break instead
of an overnight job.

## Why 2048 and not 256

Caching the LSTM's 256-d output would be 65x smaller again — and would **freeze the LSTM
permanently**, since those numbers come *out* of it. Caching at 2048 (before the LSTM)
keeps it trainable. Section 3 measures exactly what that choice is worth.

**All logic lives in `src/`** — `extract_frame_features` (`src/utils/train.py`),
`build_clip_loader` / `save_feature_cache` / `verify_cache` (`src/data/cache.py`).

## 1 — Bootstrap

In [ ]:
# ── ULAL bootstrap — self-contained, no Drive dependency ────────────────────
# One shallow git clone instead of hundreds of API calls. subprocess takes a
# LIST, never a shell string, so the '&' in the branch name is safe.
import os, sys, subprocess
from getpass import getpass

REPO, BRANCH, ROOT = "silvergjeka22/Unified-Lifelong-Action-Learning", "GAN&ActiveLearning", "/content/ulal"
token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden): ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(f"{ROOT}/.git"):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)

sys.path.insert(0, ROOT)
print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# ── Drive + dataset + paths ─────────────────────────────────────────────────
# Exports ULAL_* env vars that config.py reads. Nothing is written to disk and
# no kernel restart is needed.
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

## 2 — Label space

In [ ]:
# ── Global label space (src/data/cache.py) ──────────────────────────────────
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]     # Sections 4-5 stream over these
FEWSHOT_TASK = 3          # Section 7 probe only

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

## 3 — Preprocess videos into clips

Skipped when the cache exists. `splits` is passed explicitly: `splits=[]` is falsy, so
`preprocess_dataset` silently substitutes all three.

In [ ]:
FORCE_REEXTRACT = False    # True only if classes or the teacher checkpoint change

NEED = FORCE_REEXTRACT or not os.path.exists(cfg.FEAT_CACHE)
print(f"Cache: {cfg.FEAT_CACHE}")
print("  MISS -> preprocess + extract (~20-40 min, once)" if NEED
      else "  HIT  -> nothing to do; later notebooks load it in ~30 s")

if NEED:
    for root, classes, name in zip(TASK_ROOTS, TASK_CLASSES, TASK_NAMES):
        print(f"\n=== {name}: {len(classes)} classes -> {root}")
        cfg.OUTPUT_ROOT = root                # preprocess_dataset reads this at call time
        preprocess_dataset(splits=["train", "val", "test"], target_classes=classes)

## 4 — Load the teacher, freeze it

Two things are kept: the **ResNet** (becomes the extractor) and the **LSTM weights**
(saved into the cache so Section 3's two probes start from an identical initialisation —
the only variable between them is whether the LSTM may move).

In [ ]:
teacher = None
if NEED:
    teacher = ResNet50LSTMTeacher(
        hidden_size = cfg.TEACHER_HIDDEN,
        num_classes = len(cfg.SELECTED_CLASSES),
        dropout_p   = cfg.DROPOUT_P,
    ).to(device)

    mapped = remap_teacher_checkpoint(torch.load(cfg.RESNET50_PATH, map_location=device))
    missing, unexpected = teacher.load_state_dict(mapped, strict=False)
    real_missing = [k for k in missing if "num_batches_tracked" not in k]
    if real_missing:
        raise RuntimeError(f"Checkpoint missing real keys: {real_missing[:5]}")

    for p in teacher.parameters():
        p.requires_grad_(False)
    teacher.eval()

    print("Teacher loaded and FROZEN")
    print(f"  ResNet backbone : {sum(p.numel() for p in teacher.backbone.parameters()):>12,}  <- extractor")
    print(f"  LSTM            : {sum(p.numel() for p in teacher.lstm.parameters()):>12,}  <- saved for Section 3")
    print(f"  unexpected keys ignored: {len(unexpected)}")

## 5 — Extract

Splits are cached **separately** — train fits, val selects, test reports, never mixed.

**Shape walk, one batch of 8 clips:**
```
(8, 16, 3, 224, 224)   input
(128, 3, 224, 224)     view(B*T, ...) — the ResNet sees frames, not clips
(128, 2048, 1, 1)      frozen ResNet
(8, 16, 2048)          view(B, T, 2048)   <- SAVED (fp16)
```

In [ ]:
if NEED:
    store_dtype = getattr(torch, cfg.STORE_DTYPE)
    FEATS = {}

    for t, (root, classes, name) in enumerate(zip(TASK_ROOTS, TASK_CLASSES, TASK_NAMES)):
        for split in ["train", "val", "test"]:
            ds, loader = build_clip_loader(root, classes, global_c2i, split,
                                           batch_size=cfg.BATCH_SIZE,
                                           num_workers=cfg.NUM_WORKERS,
                                           pin_memory=(device.type == "cuda"))
            if len(ds) == 0:
                print(f"  WARNING: {name}/{split} empty at {root}/{split}")
                continue
            f, y = extract_frame_features(teacher.backbone, loader, device,
                                          backbone_dim=cfg.BACKBONE_DIM, dtype=store_dtype)
            FEATS[f"t{t}_{split}"] = (f, y)
            print(f"  {name:<6} {split:<5}: {str(tuple(f.shape)):<22}"
                  f"{f.element_size()*f.nelement()/1e6:6.1f} MB")

    save_feature_cache(cfg.FEAT_CACHE, FEATS, teacher.lstm.state_dict(),
                       cfg.BACKBONE_DIM, cfg.CLIP_LEN, global_classes, TASK_NAMES)

    del teacher
    teacher = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("ResNet released. Video is never touched again.")

## 6 — Verify

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

for k in sorted(CACHE):
    f, y = CACHE[k]
    print(f"  {k:<12}: {str(tuple(f.shape)):<22} {len(set(y.tolist())):>2} classes")

verify_cache(CACHE, TASK_CLASSES, global_c2i, cfg.BACKBONE_DIM, cfg.CLIP_LEN)
assert LSTM_STATE is not None, "lstm_state missing — Section 3 probes need it"
print("lstm_state present — Section 3 can seed both probes identically.")

---

# 7 — Is the representation any good? ⭐

**The question this answers**

> The ResNet was fine-tuned on 10 *sports* classes and then frozen. Do its features
> still carry the NEW classes (makeup, boxing), or did fine-tuning throw that away?

A **probe** answers it in seconds instead of waiting for Section 3.

We mean-pool each clip `(16, 2048) -> (2048,)` and ask a dumb classifier to separate the
classes. Pooling deliberately ignores temporal order: this measures what the **frozen
backbone** provides, independent of the LSTM that Section 3 studies.

| Probe | What it means |
|---|---|
| **1-NN** | pure geometry — are same-class clips nearest neighbours? No training at all. |
| **linear** | how much *linearly separable* information is in the features |

**How to read it**

- Every group well above chance → the frozen features carry all the classes. Any later
  accuracy drop is **forgetting**, not a bad representation.
- A group near chance → the backbone genuinely lost those classes. Stop and fix that
  first; no CL method rescues it.
- JOINT much lower than each task alone → classes are separable in isolation but
  **collide when mixed**. That is where confusions live.

In [ ]:
report = probe_report(CACHE, TASK_NAMES, k=1)

## 8 — Cluster tightness

Silhouette score in `[-1, 1]`: how tight each class cluster is versus how far apart the
classes are. Near 0 means clusters overlap heavily.

In [ ]:
task_ids = [t for t in range(len(TASK_NAMES)) if f"t{t}_train" in CACHE]

print(f"{'Group':<10}{'silhouette':>12}")
print("-" * 22)
for t in task_ids:
    s = separability(CACHE, [t])
    if s is not None:
        print(f"{TASK_NAMES[t]:<10}{s:>12.3f}")
s_joint = separability(CACHE, task_ids)
print("-" * 22)
print(f"{'JOINT':<10}{s_joint:>12.3f}   <- all classes together")
print("\n(higher = tighter, better-separated clusters)")

## 9 — Which classes will fight?

Distance between class centroids. Classes sitting close together occupy the same region
of feature space — **those are the pairs that will compete during continual learning.**

This plot predicts your confusions *before* you train anything.

In [ ]:
cents        = class_centroids(CACHE, task_ids, "train")
labels, dist = centroid_distances(cents)

print("10 closest class pairs (most likely to be confused):\n")
for d, a, b in closest_pairs(cents, global_classes, top=10):
    ta = next(i for i, g in enumerate(TASK_CLASSES) if a in g)
    tb = next(i for i, g in enumerate(TASK_CLASSES) if b in g)
    flag = "  <-- DIFFERENT TASKS" if ta != tb else ""
    print(f"  {d:7.2f}   {a:<22} <-> {b:<22} [{TASK_NAMES[ta]}/{TASK_NAMES[tb]}]{flag}")

## 10 — Plots

**(a)** probe accuracy vs chance — the headline numbers
**(b)** latent space coloured by **task** — do the tasks occupy separate regions?
**(c)** latent space coloured by **class** — do individual classes form clusters?
**(d)** centroid distance heatmap — which classes are neighbours

In [ ]:
task_of = {global_c2i[c]: t for t, g in enumerate(TASK_CLASSES) for c in g}

print("Projecting to 2-D (PCA -> t-SNE), this takes ~30 s ...")
xy, lab = project_2d(CACHE, task_ids, split="train", method="tsne", seed=cfg.SEED)

fig = plt.figure(figsize=(16, 11))
gs  = fig.add_gridspec(2, 2, hspace=0.25, wspace=0.2)

plot_probe_bars(report, ax=fig.add_subplot(gs[0, 0]))
plot_latent_space(xy, lab, global_classes, task_of=task_of,
                  title="Latent space coloured by TASK\n(overlap = tasks will compete)",
                  ax=fig.add_subplot(gs[0, 1]))
plot_latent_space(xy, lab, global_classes,
                  title="Latent space coloured by CLASS\n(tight clusters = good features)",
                  ax=fig.add_subplot(gs[1, 0]))
plot_centroid_heatmap(labels, dist, global_classes, ax=fig.add_subplot(gs[1, 1]))

fig.suptitle("Frozen-embedding diagnostics — before any training",
             fontsize=13, fontweight="bold")
plt.savefig(f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {cfg.RESULTS_DIR}/stage2_embedding_diagnostics.png")

## 11 — Save the diagnostics

In [ ]:
diag = {
    "stage": "2_embedding_quality",
    "probe": report,
    "silhouette": {TASK_NAMES[t]: separability(CACHE, [t]) for t in task_ids},
    "silhouette_joint": s_joint,
    "closest_pairs": [{"dist": float(d), "a": a, "b": b}
                      for d, a, b in closest_pairs(cents, global_classes, top=10)],
    "n_clips": int(sum(f.shape[0] for f, _ in CACHE.values())),
    "backbone_dim": cfg.BACKBONE_DIM,
    "clip_len": cfg.CLIP_LEN,
}
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage2_embedding_diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)

print(f"JOINT linear probe : {report['JOINT']['linear']:.2%}")
print(f"JOINT 1-NN         : {report['JOINT']['knn']:.2%}")
print(f"chance             : {report['JOINT']['chance']:.2%}")
print(f"\nSaved -> {cfg.RESULTS_DIR}/stage2_embedding_diagnostics.json")

---

## Done — next: Section 3

**Check the extraction:** every task/split has a sensible clip count (no empty
warnings), shapes are `(N, 16, 2048)`, and `Task4` is cached too (Section 7 needs it;
it's free now).

**Check the probe (section 7 above) — this is the important one:**

| JOINT linear probe | Meaning | Do |
|---|---|---|
| **> 70%** | features are good; representation is not the bottleneck | continue |
| **40-70%** | usable but strained; expect the Section 3 ceiling near here | continue, note it |
| **< 40%** | features are weak for these classes | **stop** — uncomment classes in `config.py` for a more diverse base, or reconsider the backbone |

The probe is a **preview of the Section 3 ceiling**. If the joint probe is 70%, do not
expect a CL arm to beat that.

**Re-running:** later notebooks just load this cache (~30 s). Only
`FORCE_REEXTRACT = True` redoes it.

**Next:** `3_freeze_study/3.0_freeze_boundary.ipynb` — trains on all classes at once with
the LSTM locked vs unlocked. Decides the architecture and gives the **ceiling** every
later number is read against.